# Emotion Vectors — ストーリー生成 (Kaggle版)

Kaggle の T4×2（週30h無料）で実行する。  
Colabと違い**セッション12時間**まで、週30hの枠がある。

## 実行手順
1. 右上 Accelerator → **GPU T4×2** を選択
2. Add-ons → Secrets → `HF_TOKEN` を登録
3. 全セルを順番に実行

---

In [ ]:
# Cell 1: GPU確認
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU が見つかりません。Accelerator → GPU T4×2 を選択してください。")

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU{i}: {props.name} / VRAM {props.total_memory / 1e9:.1f}GB")

N_GPUS = torch.cuda.device_count()
print(f"\n使用GPU数: {N_GPUS}")

In [ ]:
# Cell 2: パッケージインストール
!pip install -q 'transformers>=4.40' accelerate tqdm huggingface_hub

In [ ]:
# Cell 3: HuggingFace ログイン & 保存先設定
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from pathlib import Path
import json, os

# Kaggle Secrets から HF_TOKEN を取得
# Add-ons → Secrets → HF_TOKEN に登録しておくこと
try:
    secret = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=secret, add_to_git_credential=False)
    print("HuggingFace ログイン成功")
except Exception as e:
    print(f"警告: {e}")
    print("Add-ons → Secrets → HF_TOKEN を登録してください")

# 保存先: /kaggle/working/ はセッション中は残るが終了で消える
# → Kaggle Dataset として保存するか、セッション中にzipでダウンロード
STORIES_DIR = Path('/kaggle/working/stories')
STORIES_DIR.mkdir(parents=True, exist_ok=True)

existing = list(STORIES_DIR.rglob('*.json'))
print(f"保存先      : {STORIES_DIR}")
print(f"既存ファイル: {len(existing)} 件")

In [ ]:
# Cell 4: 実験設定
EMOTIONS = [
    "excited",    "thrilled",    "ecstatic",  "jubilant",
    "calm",       "peaceful",    "content",   "serene",
    "angry",      "panicked",    "desperate", "terrified",
    "sad",        "depressed",   "gloomy",    "lonely",
    "surprised",  "overwhelmed", "nostalgic", "guilty",
]

TOPICS = [
    "A student learns their scholarship application was denied",
    "A person runs into their ex at a mutual friend's wedding",
    "Two friends both apply for the same job",
    "A traveler's flight is delayed, causing them to miss an important event",
    "Someone finds their childhood teddy bear at a yard sale",
    "A person finds out they were adopted through a DNA test",
    "Someone receives an apology letter years after the incident",
    "A musician hears their song being performed by someone else",
    "Someone discovers their teenage diary has been published online",
    "Two strangers realize they've been dating the same person",
]

N_STORIES_PER_TOPIC = 5
print(f"感情数           : {len(EMOTIONS)}")
print(f"トピック数       : {len(TOPICS)}")
print(f"目標ストーリー数 : {len(EMOTIONS) * len(TOPICS) * N_STORIES_PER_TOPIC}")

In [ ]:
# Cell 5: モデルロード（GPU0に配置）
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "google/gemma-2-2b-it"

print(f"ロード中: {MODEL_NAME} ...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
)
model.eval()

print(f"ロード完了 : {time.time() - t0:.1f}s")
print(f"VRAM使用  : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB (GPU0)")

In [ ]:
# Cell 6: ストーリー生成関数
import re

STORY_PROMPT = """\
Write {n} different stories based on the following premise.

Topic: {topic}

The story should follow a character who is feeling {emotion}.

Format the stories like so:

[story 1]

[story 2]

[story 3]

etc.

The paragraphs should each be a fresh start, with no continuity. Try to make them diverse and not use the same turns of phrase. Across the different stories, use a mix of third-person narration and first-person narration.

IMPORTANT: You must NEVER use the word '{emotion}' or any direct synonyms of it in the stories. Instead, convey the emotion ONLY through:

- The character's actions and behaviors
- Physical sensations and body language
- Dialogue and tone of voice
- Thoughts and internal reactions
- Situational context and environmental descriptions

The emotion should be clearly conveyed to the reader through these indirect means, but never explicitly named.\
"""


def split_stories(raw: str, n: int, min_chars: int = 150) -> list:
    parts = re.split(
        r"(?:\[story\s*\d+\]"
        r"|\*\*?#{1,3}\s+[^\n]+"
        r"|#{2,3}\s+[^\n]+"
        r"|\*\*Story\s*\d+\*\*"
        r"|\*{3,}|---+)",
        raw, flags=re.IGNORECASE,
    )
    return [p.strip() for p in parts if len(p.strip()) >= min_chars][:n]


def generate_stories(emotion: str, topic: str, n: int) -> list:
    prompt = STORY_PROMPT.format(n=n, topic=topic, emotion=emotion)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=True,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return split_stories(raw, n)


print("関数定義完了")

In [ ]:
# Cell 7: ストーリー生成メインループ
# セッション終了前に Cell 8 でzipダウンロードすること
from tqdm.notebook import tqdm

t_start = time.time()
done = skipped = failed = 0
tasks = [(e, i, t) for e in EMOTIONS for i, t in enumerate(TOPICS)]

with tqdm(total=len(tasks), desc="生成中") as pbar:
    for emotion, topic_idx, topic in tasks:
        emo_dir = STORIES_DIR / emotion
        emo_dir.mkdir(exist_ok=True)
        out_path = emo_dir / f"topic{topic_idx:02d}.json"

        pbar.set_postfix(emotion=emotion[:8], topic=topic_idx)

        if out_path.exists():
            skipped += 1
            done += 1
            pbar.update(1)
            continue

        try:
            stories = generate_stories(emotion, topic, N_STORIES_PER_TOPIC)
            out_path.write_text(json.dumps({
                "emotion": emotion,
                "topic": topic,
                "topic_idx": topic_idx,
                "stories": stories,
            }, ensure_ascii=False, indent=2))
            done += 1
        except Exception as ex:
            print(f"\n[ERROR] {emotion}/topic{topic_idx:02d}: {ex}")
            failed += 1
            done += 1

        pbar.update(1)

elapsed = time.time() - t_start
print(f"\n生成: {done - skipped - failed}件  スキップ: {skipped}件  エラー: {failed}件")
print(f"経過: {elapsed/60:.1f}分")

In [ ]:
# Cell 8: 進捗確認 & zipダウンロード
# セッション終了前に必ず実行してデータを保存すること
import shutil
from pathlib import Path

total_target = len(EMOTIONS) * len(TOPICS)
completed = list(STORIES_DIR.rglob('*.json'))
print(f"完了: {len(completed)} / {total_target} ({len(completed)/total_target*100:.1f}%)")
print()
for emotion in EMOTIONS:
    emo_dir = STORIES_DIR / emotion
    count = len(list(emo_dir.glob('*.json'))) if emo_dir.exists() else 0
    bar = '█' * count + '░' * (len(TOPICS) - count)
    print(f"  {emotion:14s} [{bar}] {count}/{len(TOPICS)}")

# zipに圧縮してダウンロード
zip_path = '/kaggle/working/stories.zip'
shutil.make_archive('/kaggle/working/stories', 'zip', '/kaggle/working', 'stories')
print(f"\nzip作成完了: {zip_path}")
print("右サイドバー Output → stories.zip をダウンロードしてください")